# Phase 4: Baseline Modeling

## Objective
Establish baseline forecasting performance and compare
traditional regression models for day-ahead electricity
demand prediction.

## Data Split
- Training: 2021–2023
- Validation: 2024
- Final Test: 2025

In [1]:
# Step 4.1: Load ML dataset

from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.metrics import mean_squared_error, mean_absolute_error

# Locate project directory
project = Path.cwd()

if not (project / "data_processed").exists():
    project = project.parent

file_path = project / "data_processed" / "pjm_rto_ml_features.csv"

ml_df = pd.read_csv(file_path)

# Convert forecast origin to datetime
ml_df["forecast_origin"] = pd.to_datetime(
    ml_df["forecast_origin"], utc=True
)

# Use Eastern local year for chronological splitting
ml_df["forecast_year"] = (
    ml_df["forecast_origin"]
    .dt.tz_convert("America/New_York")
    .dt.year
)

# Split dataset
train = ml_df[ml_df["forecast_year"] <= 2023].copy()
val = ml_df[ml_df["forecast_year"] == 2024].copy()
test = ml_df[ml_df["forecast_year"] == 2025].copy()

print("Training:", train.shape)
print("Validation:", val.shape)
print("Test:", test.shape)

print("\nForecast origins:")
print("Training:", train["forecast_origin"].nunique())
print("Validation:", val["forecast_origin"].nunique())
print("Test:", test["forecast_origin"].nunique())

Training: (26112, 22)
Validation: (8784, 22)
Test: (8760, 22)

Forecast origins:
Training: 1088
Validation: 366
Test: 365


In [2]:
# Step 4.2: Seasonal Naive Baseline

# Validation set actual values
y_val = val["mw"]

# Predict each hour using load from 24 hours earlier
naive_pred = val["lag_24"]

# Evaluate prediction accuracy
naive_rmse = np.sqrt(
    mean_squared_error(y_val, naive_pred)
)

naive_mae = mean_absolute_error(
    y_val, naive_pred
)

print("Seasonal Naive Baseline — Validation 2024")
print(f"RMSE: {naive_rmse:,.2f} MW")
print(f"MAE:  {naive_mae:,.2f} MW")

Seasonal Naive Baseline — Validation 2024
RMSE: 6,422.94 MW
MAE:  4,799.88 MW


In [3]:
# Initialize model comparison table

results = pd.DataFrame([
    {
        "Model": "Seasonal Naive",
        "RMSE": naive_rmse,
        "MAE": naive_mae
    }
])

display(results.round(2))

,Model,RMSE,MAE
0,Seasonal Naive,6422.94,4799.88


In [4]:
# Step 4.3: Linear Regression Baseline

from sklearn.linear_model import LinearRegression

# Define feature groups
calendar_features = [
    "hour", "day_of_week", "month",
    "is_weekend", "horizon"
]

lag_features = [
    "lag_24", "lag_48", "lag_168"
]

rolling_features = [
    "rolling_mean_24", "rolling_std_24",
    "rolling_mean_168", "rolling_std_168"
]

features = calendar_features + lag_features + rolling_features

# Prepare training and validation data
X_train = train[features]
y_train = train["mw"]

X_val = val[features]
y_val = val["mw"]

# Train Linear Regression
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)

# Predict validation data
lr_pred = lr_model.predict(X_val)

# Evaluate
lr_rmse = np.sqrt(mean_squared_error(y_val, lr_pred))
lr_mae = mean_absolute_error(y_val, lr_pred)

print("Linear Regression — Validation 2024")
print(f"RMSE: {lr_rmse:,.2f} MW")
print(f"MAE:  {lr_mae:,.2f} MW")

Linear Regression — Validation 2024
RMSE: 5,660.35 MW
MAE:  4,295.51 MW


In [5]:
# Update model comparison table

results = pd.concat([
    results,
    pd.DataFrame([{
        "Model": "Linear Regression",
        "RMSE": lr_rmse,
        "MAE": lr_mae
    }])
], ignore_index=True)

display(results.round(2))

,Model,RMSE,MAE
0,Seasonal Naive,6422.94,4799.88
1,Linear Regression,5660.35,4295.51


In [6]:
# Step 4.4: Ridge and Lasso Regression

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge, Lasso

regularized_models = {
    "Ridge Regression": make_pipeline(
        StandardScaler(),
        Ridge(alpha=1.0)
    ),
    "Lasso Regression": make_pipeline(
        StandardScaler(),
        Lasso(alpha=1.0, max_iter=20000)
    )
}

for name, model in regularized_models.items():

    # Train
    model.fit(X_train, y_train)

    # Predict
    pred = model.predict(X_val)

    # Evaluate
    rmse = np.sqrt(mean_squared_error(y_val, pred))
    mae = mean_absolute_error(y_val, pred)

    results = pd.concat([
        results,
        pd.DataFrame([{
            "Model": name,
            "RMSE": rmse,
            "MAE": mae
        }])
    ], ignore_index=True)

    print(f"{name}")
    print(f"RMSE: {rmse:,.2f} MW")
    print(f"MAE:  {mae:,.2f} MW\n")

display(results.round(2))

Ridge Regression
RMSE: 5,660.19 MW
MAE:  4,295.42 MW

Lasso Regression
RMSE: 5,658.35 MW
MAE:  4,293.93 MW



,Model,RMSE,MAE
0,Seasonal Naive,6422.94,4799.88
1,Linear Regression,5660.35,4295.51
2,Ridge Regression,5660.19,4295.42
3,Lasso Regression,5658.35,4293.93


### Baseline Model Comparison

All three regression models outperform the seasonal naive baseline on the 2024 validation set. Lasso achieves the lowest RMSE (5,658.35 MW), representing an approximately 11.9% reduction compared with the naive baseline. However, the differences among Linear Regression, Ridge, and Lasso are minimal, suggesting that regularization provides limited improvement with the current feature set and initial hyperparameters.